# Project and Branches

## Example: Create a project with several branches

#### Set things up

Required imports:

In [2]:
from odeon.model import (
    Project,
    Branch,
    Building,
)

#### Create a Project `my_project`

The project is a container object for anything we want to model.

In [3]:
my_project = Project(name="Example project to show branch behaviour")

#### Create a root Branch 

By using Branches, we can store several versions of our projects in parallel and keep track of relations. The main branch should be used to represent a base scenario or the status quo:

In [4]:
my_base_scenario = Branch(name="base", year=2022)
my_project.main_branch = my_base_scenario

Add two buildings to our branch:

In [5]:
my_building_1 = Building(name="my_building_1")
my_base_scenario.add_objects(my_building_1)

my_building_2 = Building(name="my_building_2", branch=my_base_scenario) # another way

In a project, objects will receive unique ids:

In [6]:
print(my_building_1)
print(my_building_2)

Building(id=2, name='my_building_1')
Building(id=5, name='my_building_2')


#### Create more Branches

Additional branches could represent different scenarios, variants etc.
By `project.bifurcate_from_root()`, we create a linked copy of the root branch of a project:

In [7]:
extreme_summer_branch = my_project.bifurcate_from_main()
extreme_summer_branch.name = "future, extreme summer"
extreme_summer_branch.year = 2035

creating branch: 100%|██████████| 2/2 [00:00<00:00, 1002.46it/s]


Let's create another branch from the just created one:

In [8]:
extreme_winter_branch = my_project.bifurcate_from_branch(extreme_summer_branch)
extreme_winter_branch.name = "future, extreme winter"

creating branch: 100%|██████████| 2/2 [00:00<00:00, 2029.67it/s]


In a branch, we can add new objects that are present only in that branch (and, as a copy, in all further branches we might create from this one):

In [9]:
variant_branch = my_project.bifurcate_from_branch(source_branch = extreme_summer_branch)
variant_branch.name="another future variant"

my_optional_building = Building(name="my_optional_building")
variant_branch.add_objects(my_optional_building)

creating branch: 100%|██████████| 2/2 [00:00<00:00, 1005.23it/s]


It is possible to track the reference object of an object in a bifurcated branch:

In [10]:
for o in variant_branch.objects:
    print(f"{o} --> {variant_branch.get_reference(o)}")

Building(id=22, name='my_building_1') --> Building(id=8, name='my_building_1')
Building(id=25, name='my_building_2') --> Building(id=11, name='my_building_2')
Building(id=29, name='my_optional_building') --> None


#### Inspect results

In [17]:
from odeon.processing.prints import print_project_stats, print_object_tree

print("Project Overview:\n")
print_project_stats(my_project)

print("\nBranch overview:\n")
for branch in my_project.branches:
    print_object_tree(branch)
    print("")

Project Overview:

└─<Project>: Example project to show branch behaviour
  │ 
  └─<Project>.branch[0](main): base
    │ - description = []
    │ - validity year = 2022
    │ - no. of base objects = 2
    │ - no. of objects = 6
    │ 
    └─<Project>.branches[1]: future, extreme summer
      │ - description = []
      │ - validity year = 2035
      │ - no. of base objects = 2
      │ - no. of objects = 6
      │ 
      ├─<Project>.branches[2]: future, extreme winter
      │   - description = []
      │   - validity year = 2035
      │   - no. of base objects = 2
      │   - no. of objects = 6
      │   
      └─<Project>.branches[3]: another future variant
          - description = []
          - validity year = 2035
          - no. of base objects = 3
          - no. of objects = 9
          

Branch overview:

└─Branch(id=0, n_objects=2, year=2022, name='base')
  ├─Building(id=2, name='my_building_1')
  │ ├─FootprintNominalBuildingGeometry(id=1)
  │ └─EnergySystem(id=3)
  └─Building(i